<a href="https://colab.research.google.com/github/bastow-libby/379-Final-Project/blob/main/Final_Project_379.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [185]:
import pandas as pd
import sqlite3
import glob
from datetime import datetime

# Suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

Extraction

In [186]:
# Stages of the ETL process are logged into this file
log_file = 'etl_log_file.txt'

# The names of the sqlite database and table to created to store the cleaned data
db_name = 'hr.db'
table_name = 'employees'

Examine the contents of the different types of files to see the formatting for extraction

In [187]:
# load one csv to discover its content, columns, etc.
df = pd.read_csv('ids_0.csv')

# display the data types of the columns
print(df.dtypes)

# display the head of the dataframe
df.head()

# print the columns
print(df.columns)

 Destination Port                 int64
 Flow Duration                    int64
 Total Fwd Packets                int64
 Total Backward Packets           int64
Total Length of Fwd Packets       int64
 Total Length of Bwd Packets      int64
 Fwd Packet Length Max            int64
 Fwd Packet Length Min            int64
 Fwd Packet Length Mean         float64
 Fwd Packet Length Std          float64
Bwd Packet Length Max             int64
 Bwd Packet Length Min            int64
 Bwd Packet Length Mean         float64
 Bwd Packet Length Std          float64
Flow Bytes/s                    float64
 Flow Packets/s                 float64
 Flow IAT Mean                  float64
 Flow IAT Std                   float64
 Flow IAT Max                     int64
 Flow IAT Min                     int64
Fwd IAT Total                     int64
 Fwd IAT Mean                   float64
 Fwd IAT Std                    float64
 Fwd IAT Max                      int64
 Fwd IAT Min                      int64


In [188]:
# load one json to discover its content, columns, etc.
df = pd.read_json('ids_4.json', lines=True)

# display the data types of the columns
print(df.dtypes)

# display the head of the dataframe
df.head()

 Destination Port                 int64
 Flow Duration                    int64
 Total Fwd Packets                int64
 Total Backward Packets           int64
Total Length of Fwd Packets       int64
 Total Length of Bwd Packets      int64
 Fwd Packet Length Max            int64
 Fwd Packet Length Min            int64
 Fwd Packet Length Mean         float64
 Fwd Packet Length Std          float64
Bwd Packet Length Max             int64
 Bwd Packet Length Min            int64
 Bwd Packet Length Mean         float64
 Bwd Packet Length Std          float64
Flow Bytes/s                    float64
 Flow Packets/s                 float64
 Flow IAT Mean                  float64
 Flow IAT Std                   float64
 Flow IAT Max                     int64
 Flow IAT Min                     int64
Fwd IAT Total                     int64
 Fwd IAT Mean                   float64
 Fwd IAT Std                    float64
 Fwd IAT Max                      int64
 Fwd IAT Min                      int64


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,27887,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,53,181,2,2,74,154,37,37,37.0,0.0,...,44,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,80,115,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,80,13,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,56529,50,1,1,0,0,0,0,0.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [189]:
# load one parquet to discover its content, columns, etc.
df = pd.read_parquet('ids_5.parquet')

# display the data types of the columns
print(df.dtypes)

# display the head of the dataframe
df.head()

 Destination Port                 int64
 Flow Duration                    int64
 Total Fwd Packets                int64
 Total Backward Packets           int64
Total Length of Fwd Packets       int64
 Total Length of Bwd Packets      int64
 Fwd Packet Length Max            int64
 Fwd Packet Length Min            int64
 Fwd Packet Length Mean         float64
 Fwd Packet Length Std          float64
Bwd Packet Length Max             int64
 Bwd Packet Length Min            int64
 Bwd Packet Length Mean         float64
 Bwd Packet Length Std          float64
Flow Bytes/s                    float64
 Flow Packets/s                 float64
 Flow IAT Mean                  float64
 Flow IAT Std                   float64
 Flow IAT Max                     int64
 Flow IAT Min                     int64
Fwd IAT Total                     int64
 Fwd IAT Mean                   float64
 Fwd IAT Std                    float64
 Fwd IAT Max                      int64
 Fwd IAT Min                      int64


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,14258,2,2,12,0,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,DoS Hulk
1,80,116312384,6,6,423,11595,411,0,70.500000,166.836147,...,20,6999.0,0.0,6999,6999,58200000.0,59600000.0,100000000,16000000,DoS Hulk
2,80,3,2,0,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,DoS Hulk
3,80,84082357,9,7,366,11595,366,0,40.666667,122.000000,...,32,995.0,0.0,995,995,83900000.0,0.0,83900000,83900000,DoS Hulk
4,80,2,2,0,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,DoS Hulk


Since the columns appear to be the same, we will extract the data and then clean it

In [190]:
def extract() -> pd.DataFrame:
    """This function combines all the csv and json files into a dataframe

    Args:
        None: The function reads all csv and json files in the working directory

    Returns:
        data (pd.DataFrame): All data sources combined in a single dataframe
    """
    ## create an empty dataframe to hold all the data
    # The columns' names are given based on an earlier exploration of a csv file
    data = pd.DataFrame(columns=[' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count',
       ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count',
       ' ACK Flag Count', ' URG Flag Count', ' CWE Flag Count',
       ' ECE Flag Count', ' Down/Up Ratio', ' Average Packet Size',
       ' Avg Fwd Segment Size', ' Avg Bwd Segment Size',
       ' Fwd Header Length.1', 'Fwd Avg Bytes/Bulk', ' Fwd Avg Packets/Bulk',
       ' Fwd Avg Bulk Rate', ' Bwd Avg Bytes/Bulk', ' Bwd Avg Packets/Bulk',
       'Bwd Avg Bulk Rate', 'Subflow Fwd Packets', ' Subflow Fwd Bytes',
       ' Subflow Bwd Packets', ' Subflow Bwd Bytes', 'Init_Win_bytes_forward',
       ' Init_Win_bytes_backward', ' act_data_pkt_fwd',
       ' min_seg_size_forward', 'Active Mean', ' Active Std', ' Active Max',
       ' Active Min', 'Idle Mean', ' Idle Std', ' Idle Max', ' Idle Min',
       ' Label'])

    ## glob.glob('*.csv')
    #
    for csvfile in glob.glob('*.csv'):
        # load the current csv file into a temporary dataframe
        tmp_df = pd.read_csv(csvfile)

        # concatenate the loaded data with the data frame
        data = pd.concat([data, tmp_df], ignore_index=True)

    ## glob.glob('*.json')

    for jsonfile in glob.glob('*.json'):
        # load the current json file into a temporary dataframe
        tmp_df = pd.read_json(jsonfile, lines=True)

        # concatenate the loaded data to the data frame
        data = pd.concat([data, tmp_df], ignore_index=True)

    ## glob.glob ('*parquet')
    for parquetfile in glob.glob('*.parquet'):
        # load the current parquet file into a temporary dataframe
        tmp_df = pd.read_parquet(parquetfile)

        # concatenate the loaded data to the data frame
        data = pd.concat([data, tmp_df], ignore_index=True)

    # return the combined (extracted) data
    return data

In [191]:
data = extract()
print(data.dtypes)
print('data shape:', data.shape)
print('column names:', data.columns)

 Destination Port                object
 Flow Duration                   object
 Total Fwd Packets               object
 Total Backward Packets          object
Total Length of Fwd Packets      object
 Total Length of Bwd Packets     object
 Fwd Packet Length Max           object
 Fwd Packet Length Min           object
 Fwd Packet Length Mean         float64
 Fwd Packet Length Std          float64
Bwd Packet Length Max            object
 Bwd Packet Length Min           object
 Bwd Packet Length Mean         float64
 Bwd Packet Length Std          float64
Flow Bytes/s                    float64
 Flow Packets/s                 float64
 Flow IAT Mean                  float64
 Flow IAT Std                   float64
 Flow IAT Max                    object
 Flow IAT Min                    object
Fwd IAT Total                    object
 Fwd IAT Mean                   float64
 Fwd IAT Std                    float64
 Fwd IAT Max                     object
 Fwd IAT Min                     object


Transform Stage

First, we will identify any NaNs

In [192]:
from math import nan
import math
#Adjust the display function to see all columns
pd.options.display.max_rows = 500

print(data.isna().sum())

print(f'Total number of missing values: {data.isna().sum().sum()}')



 Destination Port                 0
 Flow Duration                    0
 Total Fwd Packets                0
 Total Backward Packets           0
Total Length of Fwd Packets       0
 Total Length of Bwd Packets      0
 Fwd Packet Length Max            0
 Fwd Packet Length Min            0
 Fwd Packet Length Mean           0
 Fwd Packet Length Std            0
Bwd Packet Length Max             0
 Bwd Packet Length Min            0
 Bwd Packet Length Mean           0
 Bwd Packet Length Std            0
Flow Bytes/s                    118
 Flow Packets/s                  33
 Flow IAT Mean                    0
 Flow IAT Std                     0
 Flow IAT Max                     0
 Flow IAT Min                     0
Fwd IAT Total                     0
 Fwd IAT Mean                     0
 Fwd IAT Std                      0
 Fwd IAT Max                      0
 Fwd IAT Min                      0
Bwd IAT Total                     0
 Bwd IAT Mean                     0
 Bwd IAT Std                

Since there are so few missing values, and we are trying to identify each packet, we will drop the rows with missing values rather than fill them in with averages.

In [193]:
data.dropna(inplace=True)

print(f'Total number of missing values: {data.isna().sum().sum()}')

Total number of missing values: 0


Now, we need to drop all heartbleed instances, as they are not DoS attacks

In [194]:
data = data[data[' Label'] != 'Heartbleed']
print('data shape:', data.shape)

data shape: (63000, 79)


Next, we need to adjust the labels for all DoS attacks so they can be boolean (grouped together under attack rather than by label)

In [195]:

data[' Label'] = data[' Label'].where(data[' Label'] == 'BENIGN', 'ATTACK')

We need to convert the values to integers now

In [196]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 63000 entries, 0 to 63128
Data columns (total 79 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0    Destination Port             63000 non-null  object 
 1    Flow Duration                63000 non-null  object 
 2    Total Fwd Packets            63000 non-null  object 
 3    Total Backward Packets       63000 non-null  object 
 4   Total Length of Fwd Packets   63000 non-null  object 
 5    Total Length of Bwd Packets  63000 non-null  object 
 6    Fwd Packet Length Max        63000 non-null  object 
 7    Fwd Packet Length Min        63000 non-null  object 
 8    Fwd Packet Length Mean       63000 non-null  float64
 9    Fwd Packet Length Std        63000 non-null  float64
 10  Bwd Packet Length Max         63000 non-null  object 
 11   Bwd Packet Length Min        63000 non-null  object 
 12   Bwd Packet Length Mean       63000 non-null  float64
 13   Bwd P

In [197]:
# Convert to int separated for clarity and cleanliness of code
def convert_to_int(x):
    return int(x)


In [198]:
for column in data.columns[:-1]: #iterate over all columns up to the last one
    if data[column].dtype == object:
        data[column] = data[column].apply(lambda x: convert_to_int(x))


In [166]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 63000 entries, 0 to 63128
Data columns (total 79 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0    Destination Port             63000 non-null  int64  
 1    Flow Duration                63000 non-null  int64  
 2    Total Fwd Packets            63000 non-null  int64  
 3    Total Backward Packets       63000 non-null  int64  
 4   Total Length of Fwd Packets   63000 non-null  int64  
 5    Total Length of Bwd Packets  63000 non-null  int64  
 6    Fwd Packet Length Max        63000 non-null  int64  
 7    Fwd Packet Length Min        63000 non-null  int64  
 8    Fwd Packet Length Mean       63000 non-null  float64
 9    Fwd Packet Length Std        63000 non-null  float64
 10  Bwd Packet Length Max         63000 non-null  int64  
 11   Bwd Packet Length Min        63000 non-null  int64  
 12   Bwd Packet Length Mean       63000 non-null  float64
 13   Bwd P

We will also look at the correlation between the columns to reduce dimensionality.

In [199]:
data.iloc[:, :-1].corr().style.background_gradient(cmap='coolwarm') #iloc method found on stack exchange

According to my research, rows/columns with NaN have 0 variance, meaning all the values are the same and can be eliminated. We also want to reduce columns that have high correlation.

In [200]:
data = data.loc[:, data.nunique() > 1] # Command to drop non-unique values from geeks for geek

In [201]:
data.iloc[:, :-1].corr().style.background_gradient(cmap='coolwarm') #iloc method found on stack exchange

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
Destination Port,1.000000,-0.119436,-0.004940,-0.003280,0.029740,-0.003293,0.051429,-0.006236,0.055451,0.059933,-0.138927,-0.014844,-0.147682,-0.135468,0.027588,0.059385,-0.062598,-0.092820,-0.109269,-0.032745,-0.113396,-0.075295,-0.090366,-0.108309,-0.036505,-0.045364,-0.039540,-0.038306,-0.039878,-0.009399,0.147893,-0.008198,-0.005216,0.051806,0.096325,-0.003289,-0.126499,-0.130963,-0.130436,-0.102656,-0.039490,0.147893,-0.000329,-0.095839,0.130197,0.351398,-0.000329,0.055065,-0.128702,0.055451,-0.147682,-0.008198,-0.004940,0.029740,-0.003280,-0.003294,-0.073333,0.170824,-0.001316,-0.088121,-0.036157,-0.010143,-0.036189,-0.034745,-0.104272,-0.023801,-0.107077,-0.101077
Flow Duration,-0.119436,1.000000,0.023815,0.021629,0.062399,0.019086,0.192499,-0.092908,0.021207,0.243194,0.296782,-0.159334,0.330723,0.234140,-0.051118,-0.284375,0.414251,0.853750,0.967867,0.155168,0.998917,0.578477,0.858140,0.967981,0.217448,0.444802,0.364041,0.427490,0.446100,0.074670,-0.104684,0.030219,0.027996,-0.281281,-0.069169,-0.123243,0.303824,0.355733,0.299110,0.199062,0.341628,-0.104684,-0.003711,-0.312092,0.145993,-0.143329,-0.003711,0.038450,0.339736,0.021207,0.330723,0.030219,0.023815,0.062399,0.021629,0.019091,-0.316105,-0.037122,0.017195,-0.282837,0.112795,0.090342,0.127138,0.095775,0.947069,0.193434,0.967199,0.921625
Total Fwd Packets,-0.004940,0.023815,1.000000,0.999225,0.421053,0.998881,0.023695,-0.004631,0.000386,0.010273,0.029666,-0.006046,0.026725,0.012625,-0.002406,-0.011810,-0.010370,0.000775,0.001608,-0.009853,0.022997,-0.010147,0.002996,0.001567,-0.011001,0.031283,0.000487,-0.000526,0.002445,0.002334,0.039702,0.999044,0.998498,-0.011647,-0.003297,-0.005754,0.029840,0.030844,0.017863,0.010696,-0.003898,0.039702,0.000059,0.011552,-0.004970,-0.007233,0.000059,0.006116,0.027870,0.000386,0.026725,0.999044,1.000000,0.421053,0.999225,0.998873,0.010915,0.003533,0.998646,-0.014438,0.008411,0.011685,0.011394,0.006280,0.000780,0.003172,0.001235,0.000412
Total Backward Packets,-0.003280,0.021629,0.999225,1.000000,0.421259,0.998718,0.022780,-0.002612,0.001036,0.009585,0.027445,-0.002682,0.024767,0.009697,-0.001340,-0.009989,-0.009285,0.001115,0.001950,-0.008418,0.021063,-0.007558,0.004031,0.001913,-0.008851,0.034260,0.000690,0.002780,0.006284,-0.001336,0.041827,0.997784,0.999516,-0.009934,-0.001780,-0.003895,0.027580,0.032304,0.016549,0.008800,0.002528,0.041827,0.000026,0.004110,-0.003731,-0.004279,0.000026,0.019009,0.029671,0.001036,0.024767,0.997784,0.999225,0.421259,1.000000,0.998716,0.003697,0.001976,0.998547,-0.021258,-0.003116,0.004095,-0.000980,-0.003771,0.002046,-0.002810,0.001597,0.002320
Total Length of Fwd Packets,0.029740,0.062399,0.421053,0.421259,1.000000,0.408172,0.356662,0.042736,0.257894,0.326563,0.053556,-0.020590,0.050351,0.039134,0.031569,-0.044618,-0.037180,0.017329,0.012740,-0.041240,0.058993,-0.028355,0.018402,0.0

In [202]:
import numpy as np

#Following code found on stack exchange to remove highly correlated columns
corr_matrix = data.iloc[:, :-1].corr().abs() #Build matrix

#Only examine the upper half of the matrix so we don't delete all columns
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features with correlation greater than 0.90
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
print(to_drop)
# Apply list of features to drop to matrix
data.drop(to_drop, axis=1, inplace=True)

[' Total Backward Packets', ' Total Length of Bwd Packets', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Max', ' Fwd IAT Min', ' Bwd IAT Std', ' Bwd IAT Max', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', ' SYN Flag Count', ' ECE Flag Count', ' Average Packet Size', ' Avg Fwd Segment Size', ' Avg Bwd Segment Size', ' Fwd Header Length.1', 'Subflow Fwd Packets', ' Subflow Fwd Bytes', ' Subflow Bwd Packets', ' Subflow Bwd Bytes', 'Init_Win_bytes_forward', ' act_data_pkt_fwd', ' Active Max', ' Active Min', 'Idle Mean', ' Idle Max', ' Idle Min']


In [203]:
data.iloc[:, :-1].corr().style.background_gradient(cmap='coolwarm') #iloc method found on stack exchange

,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Fwd IAT Std,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Min,Fwd PSH Flags,Bwd Packets/s,Min Packet Length,FIN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,Down/Up Ratio,Init_Win_bytes_backward,min_seg_size_forward,Active Mean,Active Std,Idle Std
Destination Port,1.000000,-0.119436,-0.004940,0.029740,0.051429,-0.006236,0.055451,0.059933,-0.138927,-0.014844,0.027588,0.059385,-0.062598,-0.092820,-0.090366,-0.045364,-0.039540,-0.009399,0.147893,0.096325,-0.003289,-0.039490,-0.000329,-0.095839,0.130197,0.351398,0.055065,0.170824,-0.088121,-0.036157,-0.010143,-0.023801
Flow Duration,-0.119436,1.000000,0.023815,0.062399,0.192499,-0.092908,0.021207,0.243194,0.296782,-0.159334,-0.051118,-0.284375,0.414251,0.853750,0.858140,0.444802,0.364041,0.074670,-0.104684,-0.069169,-0.123243,0.341628,-0.003711,-0.312092,0.145993,-0.143329,0.038450,-0.037122,-0.282837,0.112795,0.090342,0.193434
Total Fwd Packets,-0.004940,0.023815,1.000000,0.421053,0.023695,-0.004631,0.000386,0.010273,0.029666,-0.006046,-0.002406,-0.011810,-0.010370,0.000775,0.002996,0.031283,0.000487,0.002334,0.039702,-0.003297,-0.005754,-0.003898,0.000059,0.011552,-0.004970,-0.007233,0.006116,0.003533,-0.014438,0.008411,0.011685,0.003172
Total Length of Fwd Packets,0.029740,0.062399,0.421053,1.000000,0.356662,0.042736,0.257894,0.326563,0.053556,-0.020590,0.031569,-0.044618,-0.037180,0.017329,0.018402,0.109557,0.064792,0.069214,0.071276,0.018925,-0.000435,-0.005578,0.001649,0.068504,-0.050123,-0.014858,0.016589,0.143675,-0.033976,0.039008,0.199445,0.011143
Fwd Packet Length Max,0.051429,0.192499,0.023695,0.356662,1.000000,0.317284,0.708301,0.886234,0.380546,-0.115092,0.249554,-0.198615,-0.132231,0.227070,0.231433,0.252556,0.194632,0.001263,0.105171,0.167866,-0.034615,0.043625,0.008350,0.262064,-0.220877,-0.081989,0.099092,0.046899,-0.090448,-0.113951,0.080564,-0.058323
Fwd Packet Length Min,-0.006236,-0.092908,-0.004631,0.042736,0.317284,1.000000,0.835680,-0.082161,-0.097753,0.043400,0.713426,0.063354,-0.040044,-0.073901,-0.074270,-0.045175,-0.034289,0.002558,0.467522,0.566231,0.194967,-0.035148,-0.000392,-0.073165,0.072585,0.084594,0.132750,-0.008181,0.025987,-0.026184,-0.007712,-0.018053
Fwd Packet Length Mean,0.055451,0.021207,0.000386,0.257894,0.708301,0.835680,1.000000,0.431159,0.038186,-0.015954,0.612616,-0.040414,-0.057969,0.078404,0.027948,0.100711,0.105407,0.057742,0.383651,0.468080,0.121038,0.003602,0.002999,0.069473,-0.057654,0.024962,0.165390,0.049231,-0.016323,-0.037772,0.123512,-0.025691
Fwd Packet Length Std,0.059933,0.243194,0.010273,0.326563,0.886234,-0.082161,0.431159,1.000000,0.410521,-0.138214,-0.026252,-0.237740,-0.089355,0.324880,0.272687,0.290363,0.251159,0.012045,-0.077482,-0.058968,-0.119730,0.081986,0.007491,0.309316,-0.278538,-0.123720,0.100717,0.052686,-0.102038,-0.110059,0.098349,-0.059734
Bwd Packet Length Max,-0.138927,0.296782,0.029666,0.053556,0.380546,-0.097753,0.038186,0.410521,1.000000,-0.174066,-0.052806,-0.298069,-0.207821,0.303300,0.421701,0.204052,0.065220,-0.072377,-0.143072,-0.071657,-0.144637,0.197398,-0.002319,0.242839,-0.278868,-0.141846,0.081533,-0.073804,-0.204666,-0.201038,-0.048013,-0.073160
Bwd Packet Length Min,-0.014844,-0.159334,-0.006046,-0.020590,-0.115092,0.043400,-0.015954,-0.138214,-0.174066,1.000000,-0.007123,-0.048276,-0.072233,-0.111685,-0.131825,-0.072019,-0.037834,0.060321,-0.000441,0.008991,0.269285,-0.065095,-0.000710,-0.132157,-0.152125,-0.014289,0.206951,-0.015064,-0.080462,-0.046403,-0.011799,-0.029083


In [179]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 63000 entries, 0 to 63128
Data columns (total 34 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0    Destination Port            63000 non-null  int64  
 1    Flow Duration               63000 non-null  int64  
 2    Total Fwd Packets           63000 non-null  int64  
 3   Total Length of Fwd Packets  63000 non-null  int64  
 4    Fwd Packet Length Max       63000 non-null  int64  
 5    Fwd Packet Length Min       63000 non-null  int64  
 6    Fwd Packet Length Mean      63000 non-null  float64
 7    Fwd Packet Length Std       63000 non-null  float64
 8   Bwd Packet Length Max        63000 non-null  int64  
 9    Bwd Packet Length Min       63000 non-null  int64  
 10  Flow Bytes/s                 63000 non-null  float64
 11   Flow Packets/s              63000 non-null  float64
 12   Flow IAT Mean               63000 non-null  float64
 13   Flow IAT Std        

Now we should have a cleaned data set that is properly labeled. We can now load the data to a csv file

In [204]:
# load to a csv file
def loadcsv(data: pd.DataFrame) -> None:
    """This function loads the argument dataframe into a csv file

    Args:
        data (pd.DataFrame): extracted nd transformed dataframe
    """
    data.to_csv('packet_ids.csv', index=False)

In [205]:
loadcsv(data)

We will now split and standardize the data.

In [1]:
! pwd


/content
